# PDLC-ViT — Reproduce Experiment (Classification Branch)

**Paper:** A Multitask Learning-Based Vision Transformer for Plant Disease Localization and Classification
**Authors:** S. Hemalatha, Jai Jaganath Babu Jayachandran
**Source:** International Journal of Computational Intelligence Systems (2024) 17:188
**DOI:** https://doi.org/10.1007/s44196-024-00597-3
**Original framework:** MATLAB 2023a — reproduced here in **PyTorch**.

---

## DISCLAIMER (READ FIRST)

The original paper describes the PDLC-ViT architecture conceptually (component connectivity in Fig. 1–4) and
provides a training hyperparameter table (Table 2), **but does NOT publish the quantitative ViT specs**
(patch size, embedding dim, depth, heads, MLP dim, input size, normalization). The following values are
**reasonable choices confirmed by the user**, NOT extracted from the paper:

| Parameter | Value | Source |
|-----------|-------|--------|
| Input size | 224×224 | User-confirmed (ViT standard) |
| Patch size | 16×16 → 196 patches | User-confirmed |
| Embed dim | 384 | User-confirmed (ViT-Small) |
| Depth (encoder layers) | 6 | User-confirmed |
| Attention heads | 6 | User-confirmed |
| MLP dim | 1536 (4×384) | User-confirmed |
| Normalization | ImageNet mean/std | User-confirmed |
| Max epochs / early stop | 100 / patience=7 (val_loss) | User-confirmed |
| Co-scale / Co-attention impl. | Interpreted (see in-code notes) | NOT in paper |

**Confirmed directly from the paper (Table 2):** Optimizer=Adam, LR=0.001, StepLR(step=15, gamma=0.5),
batch=16, dropout=0.2, weight_decay(L2)=0.0005, loss=categorical cross-entropy, early stopping on val loss.

**Scope:** This notebook reproduces the **classification branch only**. The segmentation head (PlantDoc /
localization) is out-of-scope per the user's confirmed plan, so the MTL loss simplifies to `L_cls` only.
The paper's classification head MLP used a Sigmoid; with multiclass CrossEntropyLoss we output raw logits
(softmax handled internally) — documented difference.

**Important:** This notebook trains on the **user's own dataset**, not the paper's PlantVillage. `NUM_CLASSES`
is auto-detected. The paper's reported 99.97% accuracy is NOT directly comparable.


## 1. Install & Import

In [ ]:
# Install dependencies (Kaggle usually has torch/torchvision; torchinfo + openpyxl may need install)
!pip install -q torchinfo openpyxl Pillow

In [ ]:
import os, gc, time, random, copy
import numpy as np
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, Subset

import torchvision
from torchvision import datasets, transforms

from torchinfo import summary

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (accuracy_score, f1_score, recall_score,
                             classification_report, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)

## 2. Reproducibility (Seed)

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Full GPU reproducibility (Rule 9) — seed alone is not enough on GPU
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print("Seed set to", SEED)

## 3. Device Detection & Multi-GPU

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpus = torch.cuda.device_count()
print(f"Device: {device} | GPUs available: {n_gpus}")
if device.type == "cuda":
    for i in range(n_gpus):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

## 4. Config

All hyperparameters. Paper-confirmed values are tagged `# [PAPER Table 2]`; user-confirmed reproduction
choices (not in paper) are tagged `# [USER-CONFIRMED]`.

In [ ]:
# ----- Dataset -----
DATASET_PATH = "/kaggle/input/datasets/tminhhi/ot-final/dataset_smartcrop_plit_300"  # [USER-CONFIRMED]

# ----- Architecture (PDLC-ViT classification branch) -----
IMG_SIZE     = 224     # [USER-CONFIRMED] ViT standard, not in paper
PATCH_SIZE   = 16      # [USER-CONFIRMED] -> (224/16)^2 = 196 patches
EMBED_DIM    = 384     # [USER-CONFIRMED] ViT-Small
DEPTH        = 6       # [USER-CONFIRMED] encoder layers
NUM_HEADS    = 6       # [USER-CONFIRMED]
MLP_DIM      = 1536    # [USER-CONFIRMED] = 4 * EMBED_DIM
DROPOUT      = 0.2     # [PAPER Table 2]

# ----- Training -----
BATCH_SIZE     = 16        # [PAPER Table 2]
LR             = 0.001     # [PAPER Table 2]
WEIGHT_DECAY   = 0.0005    # [PAPER Table 2] L2 regularization
STEP_SIZE      = 15        # [PAPER Table 2] LR x0.5 every 15 epochs
GAMMA          = 0.5       # [PAPER Table 2]
MAX_EPOCHS     = 100       # [USER-CONFIRMED] paper uses early stopping, no fixed epoch
EARLY_PATIENCE = 7         # [USER-CONFIRMED] monitor val_loss

# ----- Split (used only if dataset has no train/val/test subfolders) -----
TRAIN_RATIO = 0.70   # [USER-CONFIRMED] 70/15/15 stratified
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# ----- Normalization -----
IMAGENET_MEAN = [0.485, 0.456, 0.406]   # [USER-CONFIRMED]
IMAGENET_STD  = [0.229, 0.224, 0.225]   # [USER-CONFIRMED]

MODEL_NAME = "PDLC-ViT"

NUM_WORKERS = 4   # Rule 13 — Kaggle T4x2 has 4 vCPU
PIN_MEMORY  = True

# Output paths
OUT_DIR = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)
print("Config loaded. Model:", MODEL_NAME)

## 5. Dataset Loading & Split

Auto-detects the directory layout:
- `train/ + val/ + test/`  → use directly
- `train/ + valid/`        → split `valid` 50/50 into val + test
- single folder (classes)  → stratified 70/15/15 split

`class_names` and `NUM_CLASSES` are auto-detected (Rule 12). Val/Test always use the baseline (no-aug)
transform; only the train split varies between Scenario 1 and 2.

In [ ]:
_TRAIN_DIR = os.path.join(DATASET_PATH, 'train')
_VAL_DIR   = os.path.join(DATASET_PATH, 'val')
_VALID_DIR = os.path.join(DATASET_PATH, 'valid')
_TEST_DIR  = os.path.join(DATASET_PATH, 'test')

def _stratified_indices(targets, train_r, val_r, seed):
    idx = np.arange(len(targets))
    train_idx, tmp_idx = train_test_split(
        idx, train_size=train_r, stratify=targets, random_state=seed)
    rel_val = val_r / (1.0 - train_r)
    val_idx, test_idx = train_test_split(
        tmp_idx, train_size=rel_val, stratify=np.array(targets)[tmp_idx], random_state=seed)
    return train_idx, val_idx, test_idx

def make_splits(transform_train, transform_eval):
    '''Return (train_ds, val_ds, test_ds, class_names).'''
    has_train = os.path.isdir(_TRAIN_DIR)
    has_val   = os.path.isdir(_VAL_DIR)
    has_valid = os.path.isdir(_VALID_DIR)
    has_test  = os.path.isdir(_TEST_DIR)

    # Mode 1: train/val/test
    if has_train and has_val and has_test:
        print("Split mode: 3 folders (train / val / test)")
        tr = datasets.ImageFolder(_TRAIN_DIR, transform=transform_train)
        va = datasets.ImageFolder(_VAL_DIR,   transform=transform_eval)
        te = datasets.ImageFolder(_TEST_DIR,  transform=transform_eval)
        return tr, va, te, tr.classes

    # Mode 2: train/valid -> split valid 50/50
    if has_train and has_valid and not has_test:
        print("Split mode: train/valid -> val+test from valid (50/50)")
        tr = datasets.ImageFolder(_TRAIN_DIR, transform=transform_train)
        full_eval = datasets.ImageFolder(_VALID_DIR, transform=transform_eval)
        n = len(full_eval); n_val = n // 2; n_test = n - n_val
        va, te = random_split(full_eval, [n_val, n_test],
                              generator=torch.Generator().manual_seed(SEED))
        return tr, va, te, tr.classes

    # Mode 3: single folder -> stratified 70/15/15
    root = _TRAIN_DIR if has_train else DATASET_PATH
    print(f"Split mode: single folder '{root}' -> stratified "
          f"{TRAIN_RATIO:.0%}/{VAL_RATIO:.0%}/{TEST_RATIO:.0%}")
    base = datasets.ImageFolder(root)  # no transform; for targets + class list
    targets = [s[1] for s in base.samples]
    tr_idx, va_idx, te_idx = _stratified_indices(targets, TRAIN_RATIO, VAL_RATIO, SEED)
    tr = Subset(datasets.ImageFolder(root, transform=transform_train), tr_idx)
    va = Subset(datasets.ImageFolder(root, transform=transform_eval), va_idx)
    te = Subset(datasets.ImageFolder(root, transform=transform_eval), te_idx)
    return tr, va, te, base.classes

def get_class_names(ds):
    # Works for both ImageFolder and Subset(ImageFolder) (Rule 12)
    return ds.dataset.classes if isinstance(ds, Subset) else ds.classes

In [ ]:
# Baseline transform (Scenario 1 train == val == test)
transform_baseline = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Probe once to discover classes / counts (eval transform on both sides here)
_tr, _va, _te, class_names = make_splits(transform_baseline, transform_baseline)
NUM_CLASSES = len(class_names)
print(f"NUM_CLASSES = {NUM_CLASSES}")
print("Classes:", class_names)
print(f"Train: {len(_tr)} | Val: {len(_va)} | Test: {len(_te)}")

## 6. Dataset Distribution & Class Imbalance (Rule 11)

In [ ]:
# Compute class distribution from the TRAIN split
def train_targets_of(ds):
    if isinstance(ds, Subset):
        all_t = np.array([s[1] for s in ds.dataset.samples])
        return all_t[ds.indices]
    return np.array([s[1] for s in ds.samples])

train_targets = train_targets_of(_tr)
counts = np.bincount(train_targets, minlength=NUM_CLASSES)

plt.figure(figsize=(max(8, NUM_CLASSES * 0.45), 5))
sns.barplot(x=list(range(NUM_CLASSES)), y=counts, color="#1F4E79")
plt.xticks(range(NUM_CLASSES), class_names, rotation=90)
plt.ylabel("Image count (train)")
plt.title(f"{MODEL_NAME} — Train Class Distribution")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "class_distribution.png"), dpi=300, bbox_inches="tight")
plt.show()

max_c, min_c = counts.max(), counts[counts > 0].min()
imbalance = max_c / min_c
print(f"Max={max_c} | Min={min_c} | Imbalance ratio={imbalance:.2f}")

USE_CLASS_WEIGHTS = imbalance >= 1.5
class_weights_tensor = None
if USE_CLASS_WEIGHTS:
    cw = compute_class_weight(class_weight="balanced",
                              classes=np.arange(NUM_CLASSES), y=train_targets)
    class_weights_tensor = torch.tensor(cw, dtype=torch.float32).to(device)
    print(f"[Class weighting] ENABLED (ratio>=1.5). weights={np.round(cw,3)}")
else:
    print("[Class weighting] DISABLED (ratio<1.5)")

## 7. PDLC-ViT Model Definition (Classification Branch)

Architecture maps to paper Fig. 1–4. Modules where the paper gives no layer spec are implemented by
**interpretation** and flagged inline with `# [INTERPRETED - not in paper]`.

Flow:
```
Image -> PatchEmbed (16x16, 196 tokens)
      -> CoScale (3x3 + 5x5 conv multi-scale tokens)        [INTERPRETED]
      -> CoAttention (inter-patch MHSA)                       [INTERPRETED]
      -> CrossAttention: branch_CS -> Z1, branch_CA -> Z2, MHCA(Z1,Z2) -> Z   [Fig.3]
      -> + CLS token + positional encoding
      -> Transformer Encoder x6 (MHSA + FFN, Add&Norm)        [Fig.2]
      -> Classification Head: LN -> MHCA(query=CLS, key/value=Z) -> LN -> MLP -> logits  [Fig.4]
```

In [ ]:
# ----------------------------- Building blocks -----------------------------
class MHSA(nn.Module):
    '''Multi-Head Self-Attention (scaled dot-product), paper Eq. (4)/(5).'''
    def __init__(self, dim, heads, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
    def forward(self, x):
        out, _ = self.attn(x, x, x)
        return out

class MHCA(nn.Module):
    '''Multi-Head Cross-Attention: query from one stream, key/value from another (Fig.3/Fig.4).'''
    def __init__(self, dim, heads, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
    def forward(self, query, kv):
        out, _ = self.attn(query, kv, kv)
        return out

class FeedForward(nn.Module):
    '''FFN: Linear -> ReLU -> Dropout -> Linear (paper: two linear + ReLU).'''
    def __init__(self, dim, hidden, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden), nn.ReLU(inplace=True),
            nn.Dropout(dropout), nn.Linear(hidden, dim), nn.Dropout(dropout))
    def forward(self, x):
        return self.net(x)


class PatchEmbed(nn.Module):
    '''Image Splitter: non-overlapping 16x16 patches -> EMBED_DIM tokens (paper Sect.4.2).'''
    def __init__(self, img_size, patch, in_ch, dim):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, dim, kernel_size=patch, stride=patch)
        self.n_patches = (img_size // patch) ** 2
    def forward(self, x):
        x = self.proj(x)                 # (B, dim, H/p, W/p)
        x = x.flatten(2).transpose(1, 2) # (B, n_patches, dim)
        return x


class CoScale(nn.Module):
    '''Co-scale Mechanism -> CS. [INTERPRETED - not in paper]
    Paper: 'captures multi-scale dependencies between patches'. Implemented as two parallel
    conv branches (3x3 and 5x5) over the image, projected to patch tokens, concat + linear fuse.'''
    def __init__(self, img_size, patch, in_ch, dim):
        super().__init__()
        self.branch3 = nn.Conv2d(in_ch, dim, kernel_size=patch, stride=patch, padding=0)        # base scale
        self.branch5 = nn.Conv2d(in_ch, dim, kernel_size=patch + 4, stride=patch, padding=2)    # wider receptive field
        self.fuse = nn.Linear(dim * 2, dim)
        self.norm = nn.LayerNorm(dim)
    def forward(self, x):
        b3 = self.branch3(x).flatten(2).transpose(1, 2)   # (B, n, dim)
        b5 = self.branch5(x).flatten(2).transpose(1, 2)   # (B, n, dim) — same n via padding
        cs = torch.cat([b3, b5], dim=-1)
        return self.norm(self.fuse(cs))


class CoAttention(nn.Module):
    '''Co-attention Mechanism -> CA. [INTERPRETED - not in paper]
    Paper: 'captures inter-patch dependencies'. Implemented as standard MHSA over patch tokens
    with Add&Norm.'''
    def __init__(self, dim, heads, dropout):
        super().__init__()
        self.mhsa = MHSA(dim, heads, dropout)
        self.norm = nn.LayerNorm(dim)
    def forward(self, x):
        return self.norm(x + self.mhsa(x))


class CrossAttentionBranch(nn.Module):
    '''One branch of Fig.3: input -> MHSA -> Add&Norm -> FFN -> Add&Norm -> Z_i.'''
    def __init__(self, dim, heads, mlp_dim, dropout):
        super().__init__()
        self.mhsa = MHSA(dim, heads, dropout)
        self.norm1 = nn.LayerNorm(dim)
        self.ff = FeedForward(dim, mlp_dim, dropout)
        self.norm2 = nn.LayerNorm(dim)
    def forward(self, x):
        x = self.norm1(x + self.mhsa(x))
        x = self.norm2(x + self.ff(x))
        return x


class CrossAttention(nn.Module):
    '''Cross-attention module (Fig.3): CS->Z1, CA->Z2, then MHCA(Z1,Z2)->Z.'''
    def __init__(self, dim, heads, mlp_dim, dropout):
        super().__init__()
        self.branch_cs = CrossAttentionBranch(dim, heads, mlp_dim, dropout)  # Z1
        self.branch_ca = CrossAttentionBranch(dim, heads, mlp_dim, dropout)  # Z2
        self.mhca = MHCA(dim, heads, dropout)                                # Z1=query, Z2=key/value
        self.norm = nn.LayerNorm(dim)
    def forward(self, cs, ca):
        z1 = self.branch_cs(cs)
        z2 = self.branch_ca(ca)
        z = self.norm(z1 + self.mhca(z1, z2))
        return z


class EncoderBlock(nn.Module):
    '''Transformer Encoder block (Fig.2): MHSA -> Add&Norm -> FFN -> Add&Norm.'''
    def __init__(self, dim, heads, mlp_dim, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.mhsa = MHSA(dim, heads, dropout)
        self.norm2 = nn.LayerNorm(dim)
        self.ff = FeedForward(dim, mlp_dim, dropout)
    def forward(self, x):
        x = x + self.mhsa(self.norm1(x))   # pre-norm residual
        x = x + self.ff(self.norm2(x))
        return x


class ClassificationHead(nn.Module):
    '''Classification Head (Fig.4), adapted for single-task (no segmentation).
    Block1: LN -> MHCA(query=CLS token, key/value=Z) -> residual.
    Block2: LN -> MLP(Linear->ReLU->Dropout->Linear(num_classes)) -> logits.
    NOTE: paper's MLP used Sigmoid; we output raw logits for CrossEntropyLoss.'''
    def __init__(self, dim, heads, num_classes, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.mhca = MHCA(dim, heads, dropout)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim), nn.ReLU(inplace=True),
            nn.Dropout(dropout), nn.Linear(dim, num_classes))  # logits (no Sigmoid)
    def forward(self, cls_token, z):
        # cls_token: (B,1,dim) ; z: (B,N,dim)
        q = self.norm1(cls_token)
        x = cls_token + self.mhca(q, z)        # residual MHCA
        x = self.norm2(x).squeeze(1)           # (B, dim)
        return self.mlp(x)                     # (B, num_classes)


class PDLCViT(nn.Module):
    def __init__(self, img_size=IMG_SIZE, patch=PATCH_SIZE, in_ch=3, dim=EMBED_DIM,
                 depth=DEPTH, heads=NUM_HEADS, mlp_dim=MLP_DIM,
                 num_classes=NUM_CLASSES, dropout=DROPOUT):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch, in_ch, dim)
        n_patches = self.patch_embed.n_patches

        self.co_scale = CoScale(img_size, patch, in_ch, dim)        # [INTERPRETED]
        self.co_attention = CoAttention(dim, heads, dropout)        # [INTERPRETED]
        self.cross_attention = CrossAttention(dim, heads, mlp_dim, dropout)  # Fig.3

        # CLS token + learnable positional encoding (over CLS + patches)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, dim))
        self.pos_drop = nn.Dropout(dropout)

        self.encoder = nn.ModuleList(
            [EncoderBlock(dim, heads, mlp_dim, dropout) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)

        self.cls_head = ClassificationHead(dim, heads, num_classes, dropout)

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        B = x.size(0)
        patches = self.patch_embed(x)            # (B, N, dim)
        cs = self.co_scale(x)                    # (B, N, dim)   [INTERPRETED]
        ca = self.co_attention(patches)          # (B, N, dim)   [INTERPRETED]
        z = self.cross_attention(cs, ca)         # (B, N, dim)   Fig.3 shared repr Z

        # Encoder operates on patch tokens (+CLS, +pos)  Fig.2
        cls = self.cls_token.expand(B, -1, -1)   # (B,1,dim)
        seq = torch.cat([cls, patches], dim=1)   # (B, N+1, dim)
        seq = self.pos_drop(seq + self.pos_embed)
        for blk in self.encoder:
            seq = blk(seq)
        seq = self.norm(seq)

        cls_out = seq[:, :1, :]                  # (B,1,dim) global rep (query for head)
        logits = self.cls_head(cls_out, z)       # MHCA(query=CLS, kv=Z) -> MLP  Fig.4
        return logits

print("PDLC-ViT model class defined.")

## 8. Training & Evaluation Helpers

In [ ]:
def build_model():
    model = PDLCViT(num_classes=NUM_CLASSES).to(device)
    if n_gpus > 1:
        model = nn.DataParallel(model)
        print(f"[Multi-GPU] DataParallel on {n_gpus} GPUs")
    return model

def make_loaders(train_ds, val_ds, test_ds):
    tl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=False)
    vl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    tel = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                     num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    return tl, vl, tel

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    loss_sum, n = 0.0, 0
    preds, gts = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss = criterion(out, y)
        loss_sum += loss.item() * x.size(0); n += x.size(0)
        preds.append(out.argmax(1).cpu().numpy())
        gts.append(y.cpu().numpy())
    preds = np.concatenate(preds); gts = np.concatenate(gts)
    acc = accuracy_score(gts, preds)
    return loss_sum / n, acc, gts, preds

def train_model(model, train_loader, val_loader, criterion, ckpt_path, tag):
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)  # [PAPER]
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=GAMMA)  # [PAPER]

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = -1.0
    best_val_loss = float("inf")
    epochs_no_improve = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        run_loss, run_correct, run_n = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            run_loss += loss.item() * x.size(0)
            run_correct += (out.argmax(1) == y).sum().item()
            run_n += x.size(0)
        tr_loss = run_loss / run_n
        tr_acc = run_correct / run_n

        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step()

        history["train_loss"].append(tr_loss); history["train_acc"].append(tr_acc)
        history["val_loss"].append(val_loss); history["val_acc"].append(val_acc)
        print(f"[{tag}] Epoch {epoch:3d}/{MAX_EPOCHS} | "
              f"train_loss={tr_loss:.4f} acc={tr_acc:.4f} | "
              f"val_loss={val_loss:.4f} acc={val_acc:.4f} | lr={optimizer.param_groups[0]['lr']:.2e}")

        # Best checkpoint by val_accuracy (Rule 10)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            _state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            torch.save(_state, ckpt_path)
            print(f"    -> saved best checkpoint (val_acc={val_acc:.4f})")

        # Early stopping on val_loss (Rule / user-confirmed patience=7)
        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= EARLY_PATIENCE:
                print(f"    -> early stopping at epoch {epoch} (no val_loss improvement for {EARLY_PATIENCE})")
                break

    return history, best_val_acc

def load_best(model, ckpt_path):
    target = model.module if isinstance(model, nn.DataParallel) else model
    target.load_state_dict(torch.load(ckpt_path, map_location=device))
    print(f"Loaded best checkpoint: {ckpt_path}")
    return model

def count_params(model):
    m = model.module if isinstance(model, nn.DataParallel) else model
    total = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, trainable

def compute_metrics(model_name, gts, preds, total_params):
    return {
        "model_name": model_name,
        "accuracy": accuracy_score(gts, preds),
        "f1_macro": f1_score(gts, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(gts, preds, average="macro", zero_division=0),
        "total_params": total_params,
    }

In [ ]:
# Plot helpers (each chart in its own cell per Rule 6 — these are reusable functions)
def plot_learning_curve(history, title, save_path):
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    ax[0].plot(history["train_loss"], label="train")
    ax[0].plot(history["val_loss"], label="val")
    ax[0].set_title(f"{title} — Loss"); ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss"); ax[0].legend()
    ax[1].plot(history["train_acc"], label="train")
    ax[1].plot(history["val_acc"], label="val")
    ax[1].set_title(f"{title} — Accuracy"); ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Accuracy"); ax[1].legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

def plot_confusion(gts, preds, labels, title, save_path):
    cm = confusion_matrix(gts, preds)
    plt.figure(figsize=(max(7, len(labels) * 0.6), max(6, len(labels) * 0.5)))
    sns.heatmap(cm, annot=(len(labels) <= 25), fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels, cbar=True)
    plt.title(title); plt.xlabel("Predicted"); plt.ylabel("True")
    plt.xticks(rotation=90); plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

---
## 9. SCENARIO 1 — Baseline (No Augmentation)

Train transform = **only** Resize + ToTensor + Normalize. Val/Test identical.

In [ ]:
# 9a. Transforms (baseline) — train == eval
train_tf_s1 = transform_baseline
eval_tf_s1  = transform_baseline

# 9b. Splits & loaders
train_s1, val_s1, test_s1, class_names = make_splits(train_tf_s1, eval_tf_s1)
NUM_CLASSES = len(class_names)
train_loader_s1, val_loader_s1, test_loader_s1 = make_loaders(train_s1, val_s1, test_s1)
print(f"S1 — Train:{len(train_s1)} Val:{len(val_s1)} Test:{len(test_s1)} | classes:{NUM_CLASSES}")

In [ ]:
# 9c. Build model
model_s1 = build_model()

In [ ]:
# 9d. Model summary (use .module if DataParallel — Rule 14)
_m = model_s1.module if isinstance(model_s1, nn.DataParallel) else model_s1
summary(_m, input_size=(1, 3, IMG_SIZE, IMG_SIZE),
        col_names=["input_size", "output_size", "num_params", "trainable"])

In [ ]:
# 9e. Loss + (optimizer/scheduler are created inside train_model)
criterion_s1 = (nn.CrossEntropyLoss(weight=class_weights_tensor)
                if USE_CLASS_WEIGHTS else nn.CrossEntropyLoss())
print("Loss:", "CrossEntropyLoss(weighted)" if USE_CLASS_WEIGHTS else "CrossEntropyLoss")

In [ ]:
# 9f. Train (best checkpoint by val_acc, early stop on val_loss)
CKPT_S1 = os.path.join(OUT_DIR, "best_model_s1.pth")
history_s1, best_acc_s1 = train_model(model_s1, train_loader_s1, val_loader_s1,
                                      criterion_s1, CKPT_S1, tag="S1")
print("Best val acc (S1):", best_acc_s1)

In [ ]:
# 9g. Reload best checkpoint before test eval (Rule 10)
model_s1 = load_best(model_s1, CKPT_S1)

In [ ]:
# 9h. Evaluation on TEST set
test_loss_s1, test_acc_s1, gts_s1, preds_s1 = evaluate(model_s1, test_loader_s1, criterion_s1)
total_params_s1, trainable_s1 = count_params(model_s1)
baseline_metrics = compute_metrics(MODEL_NAME, gts_s1, preds_s1, total_params_s1)

print(f"[S1] Test Accuracy : {baseline_metrics['accuracy']:.4f}")
print(f"[S1] F1 (macro)    : {baseline_metrics['f1_macro']:.4f}")
print(f"[S1] Recall (macro): {baseline_metrics['recall_macro']:.4f}")
print(f"[S1] Total params  : {total_params_s1:,}")
print("\n[S1] Classification Report:")
print(classification_report(gts_s1, preds_s1, target_names=class_names, zero_division=0))

In [ ]:
# 9i. Learning Curve (own cell — Rule 6)
BASELINE_LEARNING_CURVE_PATH = os.path.join(OUT_DIR, "s1_learning_curve.png")
plot_learning_curve(history_s1, "PDLC-ViT Baseline (S1)", BASELINE_LEARNING_CURVE_PATH)

In [ ]:
# 9j. Confusion Matrix (own cell — Rule 6)
BASELINE_CONFUSION_MATRIX_PATH = os.path.join(OUT_DIR, "s1_confusion_matrix.png")
plot_confusion(gts_s1, preds_s1, class_names, "PDLC-ViT Baseline (S1) — Confusion Matrix",
               BASELINE_CONFUSION_MATRIX_PATH)

In [ ]:
# 9k. Cleanup before Scenario 2 (Rule 13)
del model_s1, train_loader_s1, val_loader_s1, test_loader_s1
gc.collect()
torch.cuda.empty_cache()
print("Cleared GPU memory after Scenario 1.")

---
## 10. SCENARIO 2 — Augmented

Train transform adds 4 augmentations (Rule 3): RandomRotation, RandomResizedCrop (zoom/scale),
RandomHorizontalFlip, ColorJitter (brightness). Val/Test transform = baseline (no aug).

In [ ]:
# 10a. Augmented train transform
train_tf_s2 = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(15),                                   # Rotation
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),        # Zoom/Scale
    transforms.RandomHorizontalFlip(p=0.5),                          # Horizontal Flip
    transforms.ColorJitter(brightness=0.2),                          # Brightness
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf_s2 = transform_baseline  # val/test NO aug

# 10b. Splits & loaders (same split logic; train gets aug)
train_s2, val_s2, test_s2, class_names = make_splits(train_tf_s2, eval_tf_s2)
train_loader_s2, val_loader_s2, test_loader_s2 = make_loaders(train_s2, val_s2, test_s2)
print(f"S2 — Train:{len(train_s2)} Val:{len(val_s2)} Test:{len(test_s2)}")

In [ ]:
# 10c. Re-instantiate model (same class)
model_s2 = build_model()

In [ ]:
# 10d. Model summary
_m = model_s2.module if isinstance(model_s2, nn.DataParallel) else model_s2
summary(_m, input_size=(1, 3, IMG_SIZE, IMG_SIZE),
        col_names=["input_size", "output_size", "num_params", "trainable"])

In [ ]:
# 10e. Loss
criterion_s2 = (nn.CrossEntropyLoss(weight=class_weights_tensor)
                if USE_CLASS_WEIGHTS else nn.CrossEntropyLoss())
print("Loss:", "CrossEntropyLoss(weighted)" if USE_CLASS_WEIGHTS else "CrossEntropyLoss")

In [ ]:
# 10f. Train
CKPT_S2 = os.path.join(OUT_DIR, "best_model_s2.pth")
history_s2, best_acc_s2 = train_model(model_s2, train_loader_s2, val_loader_s2,
                                      criterion_s2, CKPT_S2, tag="S2")
print("Best val acc (S2):", best_acc_s2)

In [ ]:
# 10g. Reload best checkpoint
model_s2 = load_best(model_s2, CKPT_S2)

In [ ]:
# 10h. Evaluation on TEST set
test_loss_s2, test_acc_s2, gts_s2, preds_s2 = evaluate(model_s2, test_loader_s2, criterion_s2)
total_params_s2, trainable_s2 = count_params(model_s2)
aug_metrics = compute_metrics(MODEL_NAME, gts_s2, preds_s2, total_params_s2)

print(f"[S2] Test Accuracy : {aug_metrics['accuracy']:.4f}")
print(f"[S2] F1 (macro)    : {aug_metrics['f1_macro']:.4f}")
print(f"[S2] Recall (macro): {aug_metrics['recall_macro']:.4f}")
print(f"[S2] Total params  : {total_params_s2:,}")
print("\n[S2] Classification Report:")
print(classification_report(gts_s2, preds_s2, target_names=class_names, zero_division=0))

In [ ]:
# 10i. Learning Curve (own cell)
AUG_LEARNING_CURVE_PATH = os.path.join(OUT_DIR, "s2_learning_curve.png")
plot_learning_curve(history_s2, "PDLC-ViT Augmented (S2)", AUG_LEARNING_CURVE_PATH)

In [ ]:
# 10j. Confusion Matrix (own cell)
AUG_CONFUSION_MATRIX_PATH = os.path.join(OUT_DIR, "s2_confusion_matrix.png")
plot_confusion(gts_s2, preds_s2, class_names, "PDLC-ViT Augmented (S2) — Confusion Matrix",
               AUG_CONFUSION_MATRIX_PATH)

## 11. Comparison Summary (Scenario 1 vs Scenario 2)

In [ ]:
import pandas as pd
comp = pd.DataFrame([
    {"Scenario": "Baseline (No-Aug)",
     "Accuracy": baseline_metrics["accuracy"],
     "F1 (macro)": baseline_metrics["f1_macro"],
     "Recall (macro)": baseline_metrics["recall_macro"],
     "Params": baseline_metrics["total_params"]},
    {"Scenario": "Augmented",
     "Accuracy": aug_metrics["accuracy"],
     "F1 (macro)": aug_metrics["f1_macro"],
     "Recall (macro)": aug_metrics["recall_macro"],
     "Params": aug_metrics["total_params"]},
])
comp_display = comp.copy()
for c in ["Accuracy", "F1 (macro)", "Recall (macro)"]:
    comp_display[c] = comp_display[c].map(lambda v: f"{v:.4f}")
comp_display["Params"] = comp_display["Params"].map(lambda v: f"{v:,}")
print(comp_display.to_string(index=False))

## 12. Export results_summary.xlsx (Baseline + Augmented sheets, embedded charts)

In [ ]:
import openpyxl
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.drawing.image import Image as XLImage
from openpyxl.utils import get_column_letter

EXCEL_OUTPUT_PATH = os.path.join(OUT_DIR, "results_summary.xlsx")

HEADERS = ["STT", "Tên Model", "Accuracy", "F1-score", "Recall",
           "Learning Curve", "Confusion Matrix", "Parameters"]
COL_WIDTHS = [6, 35, 12, 12, 12, 38, 38, 18]
IMG_ROW_HEIGHT = 200

def style_header_row(ws):
    fill = PatternFill("solid", fgColor="1F4E79")
    font = Font(bold=True, color="FFFFFF", size=11)
    border = Border(bottom=Side(style="medium", color="FFFFFF"),
                    right=Side(style="thin", color="FFFFFF"))
    for col, header in enumerate(HEADERS, start=1):
        cell = ws.cell(row=1, column=col, value=header)
        cell.fill, cell.font, cell.border = fill, font, border
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

def write_result_row(ws, row_num, stt, metrics, lc_path, cm_path):
    data = [stt, metrics["model_name"],
            round(metrics["accuracy"], 4), round(metrics["f1_macro"], 4),
            round(metrics["recall_macro"], 4), "", "",
            f"{metrics['total_params']:,}"]
    for col, val in enumerate(data, start=1):
        cell = ws.cell(row=row_num, column=col, value=val)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[row_num].height = IMG_ROW_HEIGHT
    if os.path.exists(lc_path):
        img = XLImage(lc_path); img.width, img.height = 280, 190
        ws.add_image(img, f"F{row_num}")
    if os.path.exists(cm_path):
        img = XLImage(cm_path); img.width, img.height = 280, 190
        ws.add_image(img, f"G{row_num}")

def build_sheet(wb, name, metrics, lc_path, cm_path):
    ws = wb.create_sheet(title=name)
    for col, width in enumerate(COL_WIDTHS, start=1):
        ws.column_dimensions[get_column_letter(col)].width = width
    ws.row_dimensions[1].height = 30
    style_header_row(ws)
    write_result_row(ws, 2, 1, metrics, lc_path, cm_path)
    return ws

wb = Workbook(); wb.remove(wb.active)
build_sheet(wb, "Baseline", baseline_metrics,
            BASELINE_LEARNING_CURVE_PATH, BASELINE_CONFUSION_MATRIX_PATH)
build_sheet(wb, "Augmented", aug_metrics,
            AUG_LEARNING_CURVE_PATH, AUG_CONFUSION_MATRIX_PATH)
wb.save(EXCEL_OUTPUT_PATH)
print(f"[Excel] Saved: {EXCEL_OUTPUT_PATH}")
print(f"  Baseline  -> Acc={baseline_metrics['accuracy']:.4f} | F1={baseline_metrics['f1_macro']:.4f} | Recall={baseline_metrics['recall_macro']:.4f} | Params={baseline_metrics['total_params']:,}")
print(f"  Augmented -> Acc={aug_metrics['accuracy']:.4f} | F1={aug_metrics['f1_macro']:.4f} | Recall={aug_metrics['recall_macro']:.4f} | Params={aug_metrics['total_params']:,}")